In [10]:
import pandas as pd

base_df = pd.read_csv('../data/raw/amazon_reviews_2023.tsv', sep='\t')
base_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,"As anthologies go, okay. Three good novellas, ...",This anthology has three of my favourite Regen...,[],0451405315,0451405315,AG66ZNKTWD25GDYK634EWHSX5LNA,1085347355000,0,False
1,5.0,A rich source of ideas for innovation,Organizations must learn how to drive innovati...,[],0749471646,0749471646,AH3R5M3ZFIPA6MS4DPSNLG5BANWQ,1405511990000,0,False
2,4.0,Good info here,I received this e-book as a prize at LibraryTh...,[],1905367341,1905367341,AFQFIJQWP4MA543QLNV5JJTTMF3Q,1362606358000,0,False
3,4.0,Nice Book,Nice variety of Christmas selections. Good boo...,"[{'attachment_type': 'IMAGE', 'large_image_url...",1616771429,1616771429,AEMCAGOFOCANIBXIQTIA6WCA4I5A,1658608166845,0,True
4,5.0,GIFT,This was a gift.,[],193787950X,193787950X,AFZFY4USEL2IJ2KZ6J5VDJHP6POA,1556835558048,0,True


In [11]:
base_df = base_df.dropna(subset=["text", "rating"])
base_df = base_df[base_df["text"].str.strip().str.len() > 10]
base_df = base_df.drop_duplicates(subset=["text"])
base_df["text_len_words"] = base_df["text"].str.split().apply(len)
base_df = base_df[base_df["text_len_words"] < 1000]


In [12]:
def rating_to_sentiment(r):
    if r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    else:
        return "positive"

base_df["sentiment"] = base_df["rating"].apply(rating_to_sentiment)

In [13]:
base_df["text_len_chars"] = base_df["text"].str.len()
base_df["text_len_words"] = base_df["text"].str.split().apply(len)
base_df["has_exclamation"] = base_df["text"].str.contains("!").astype(int)
base_df["has_question"] = base_df["text"].str.contains("\?").astype(int)
base_df["has_ellipsis"] = base_df["text"].str.contains("\.\.\.").astype(int)

In [15]:
df_final = base_df[[
    "text",
    "rating",
    "sentiment",
    "text_len_chars",
    "text_len_words",
    "has_exclamation",
    "has_question",
    "has_ellipsis"
]]
df_final.head()

,text,rating,sentiment,text_len_chars,text_len_words,has_exclamation,has_question,has_ellipsis
0,This anthology has three of my favourite Regen...,3.0,neutral,4274,732,0,1,1
1,Organizations must learn how to drive innovati...,5.0,positive,1847,286,0,0,0
2,I received this e-book as a prize at LibraryTh...,4.0,positive,370,66,0,0,0
3,Nice variety of Christmas selections. Good boo...,4.0,positive,69,11,0,0,0
4,This was a gift.,5.0,positive,16,4,0,0,0


In [18]:
df_final = df_final.reset_index(drop=True)
df_final["sentiment"].value_counts(normalize=True)

sentiment
positive    0.840528
negative    0.084870
neutral     0.074602
Name: proportion, dtype: float64

In [19]:
df_final.to_parquet("../data/processed/dataset_clean.parquet", index=False)